# Phase 4 — The model

**Why (paper §3.6):** we use **EfficientNet-B0**, a compact CNN pretrained on ImageNet.
It's small (trains on our budget), its pretrained features help given our modest data,
and it's the strongest published baseline on PM25Vision — so our numbers are directly
comparable. Two tweaks:

1. **5-channel stem.** The first layer normally takes 3 channels (RGB). We widen it to 5
   (RGB + transmission + inverted saturation); `timm` seeds the two new channels from the
   pretrained RGB weights.
2. **Three monotone quantile heads.** We predict the 5th/50th/95th percentiles. To stop a
   "95th below the 50th" nonsense, we build them so each is the previous one **plus a
   strictly positive step** (`softplus`). They can never cross — guaranteed, not checked.

## Bootstrap — run this first

This one cell makes the notebook self-contained: it grabs the code from GitHub (if it
isn't already here), installs the libraries, connects Google Drive, and makes our `src`
modules importable. **Set `REPO_URL` to your repository's URL.** It's safe to re-run and
also works on a laptop.

In [ ]:
# === Bootstrap — RUN ME FIRST ===
REPO_URL = "https://github.com/keyaan01/pm25-visual-aq.git"   # <-- your repo

import os, sys, shutil, subprocess

# Where the clone lives: Kaggle -> /kaggle/working ; Colab/local -> current dir.
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
ON_KAGGLE = os.path.isdir("/kaggle/working")

def _valid_repo(p):   # a REAL checkout of THIS repo, not a rogue/partial `src` left in the workdir
    return (os.path.exists(os.path.join(p, "src", "ceiling.py"))
            and os.path.exists(os.path.join(p, "configs", "default.yaml")))

if not ON_KAGGLE and _valid_repo("."):   # local/Colab dev already inside the repo -> use it as-is
    REPO = os.path.abspath(".")
else:                                    # Kaggle (or not in a repo): ALWAYS re-clone fresh
    REPO = os.path.join(BASE, "pm25-visual-aq")
    os.chdir(BASE)                       # don't stand inside the dir we're about to delete
    shutil.rmtree(REPO, ignore_errors=True)   # kill any stale / partial / rogue clone
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)

os.chdir(REPO)
sys.path.insert(0, REPO)
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]                  # evict any `src` already imported from a stale location
# Fail LOUD if the checkout is incomplete (never silently import a namespace-package `src`):
assert os.path.exists(os.path.join(REPO, "src", "ceiling.py")), "clone incomplete (is Internet On?) -- src/ceiling.py missing at " + REPO

subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

_head = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("repo:", REPO, "| HEAD:", _head, "| Colab:", IN_COLAB)

In [ ]:
import torch
from src.config import load_config
from src import model as M
cfg = load_config()

net = M.build_model(cfg)
device = "cuda" if torch.cuda.is_available() else "cpu"
net = net.to(device)
print("device:", device)
print("backbone:", cfg["model"]["backbone"], "| input channels:", net.backbone.conv_stem.in_channels)
print("trainable parameters: %.2f M" % (M.count_parameters(net) / 1e6))

## Two outputs: quantiles (for intervals) + a point estimate (for accuracy)

The model returns a dict: **`quantiles`** (the three ordered bounds — for the interval) and
**`point`** (a dedicated estimate trained with Huber loss — for the accuracy numbers; see
Phase 5b). We confirm the quantiles come out **ascending** (q05 ≤ q50 ≤ q95) for any input, by
construction.

In [ ]:
net.eval()
with torch.no_grad():
    out = net(torch.randn(8, cfg["model"]["in_chans"], cfg["data"]["image_size"], cfg["data"]["image_size"]).to(device))
q = out["quantiles"]
print("output keys:", list(out.keys()))
print("quantiles shape (batch, 3):", tuple(q.shape), "| point shape (batch,):", tuple(out["point"].shape))
print("first sample (q05, q50, q95) in log-space:", [round(v, 3) for v in q[0].tolist()])
print("all samples ascending:", bool(torch.all(q[:, 1:] >= q[:, :-1])))

## What's next

The model outputs three ordered numbers in **log(AQI)** space. Phase 5 trains it with the
**pinball loss** (so each head learns its percentile) on the log target, using only the
physics-safe augmentations. **Next:** `05_train.ipynb`.